# IsotopePINN — Colab Pro 12k Training Run

**Steps:**
1. Mount Google Drive
2. Verify GPU
3. Unzip project files (auto-fixes Windows backslash paths)
4. Install dependencies
5. Launch 12,000-epoch training (float64, log every 100 epochs)
6. Download results (weights, graphs, loss CSV)

In [ ]:
# ── Cell 1: Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('✅ Google Drive mounted.')

In [ ]:
# ── Cell 2: Verify GPU ──
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'✅ GPU: {gpu}  |  VRAM: {vram:.1f} GB')
else:
    print('⚠️ NO GPU detected! Go to Runtime → Change runtime type → GPU (Premium).')
    print('   Training on CPU will be extremely slow (~50x).')

In [ ]:
# ── Cell 3: Auto-locate and unzip IsotopePINN_Project.zip ──
import os, zipfile

DRIVE_ROOT = '/content/drive/MyDrive'
PROJECT_DIR = os.path.join(DRIVE_ROOT, 'IsotopePINN')

# Search common locations first, then up to 2 levels deep
found_zip = None
for path in [
    os.path.join(DRIVE_ROOT, 'IsotopePINN_Project.zip'),
    os.path.join(PROJECT_DIR, 'IsotopePINN_Project.zip'),
    '/content/IsotopePINN_Project.zip',
]:
    if os.path.exists(path):
        found_zip = path
        break

if not found_zip:
    print("Searching Drive for 'IsotopePINN_Project.zip'...")
    for root, dirs, files in os.walk(DRIVE_ROOT):
        if root[len(DRIVE_ROOT):].count(os.sep) > 2:
            continue
        if 'IsotopePINN_Project.zip' in files:
            found_zip = os.path.join(root, 'IsotopePINN_Project.zip')
            break

if not found_zip:
    raise FileNotFoundError(
        "❌ Could not find 'IsotopePINN_Project.zip' anywhere in Google Drive.\n"
        "Upload the zip to your Drive root and re-run this cell."
    )

print(f'Found ZIP at: {found_zip}')
os.makedirs(PROJECT_DIR, exist_ok=True)

# Extract with Windows backslash path fix
print('Extracting (fixing Windows backslash paths)...')
with zipfile.ZipFile(found_zip, 'r') as zf:
    for member in zf.infolist():
        filename = member.filename.replace('\\', '/')
        dest_file = os.path.join(PROJECT_DIR, filename)
        os.makedirs(os.path.dirname(dest_file), exist_ok=True)
        if not member.is_dir() and not filename.endswith('/'):
            with zf.open(member) as src, open(dest_file, 'wb') as dst:
                dst.write(src.read())

# Verify critical files exist
required = ['train.py', 'pinn_model.py', 'ra226_ac225_transmutation.py',
            'graph_provenance.py', 'data/pinn_training_data.csv']
missing = [f for f in required if not os.path.exists(os.path.join(PROJECT_DIR, f))]
if missing:
    raise FileNotFoundError(f'❌ Missing after extraction: {missing}')

print('✅ All project files extracted and verified!')
print(f'   Project dir: {PROJECT_DIR}')
!ls -la {PROJECT_DIR}

In [ ]:
# ── Cell 4: Install only what Colab doesn't already have ──
# Colab pre-installs: torch, numpy, pandas, matplotlib, pillow, scikit-learn, joblib
# We only need scipy (for ODE solver in ra226_ac225_transmutation.py)
!pip install -q scipy>=1.11.2
print('✅ Dependencies ready.')

In [ ]:
# ── Cell 5: Launch 12,000-epoch training ──
# Environment variables control training behavior:
#   PINN_FLOAT64=1    → double-precision (critical for stiff ODE stability)
#   PINN_AMP=0        → disable mixed precision (incompatible with float64)
#   PINN_COMPILE=0    → skip torch.compile warmup (saves ~5 min startup)
#   PINN_JOINT_CHUNK=0 → full-batch joint step (best GPU utilization)
#   PINN_LOG_EVERY=100 → print progress every 100 epochs
#
# Default: 2000 pretrain + 10000 joint = 12000 total epochs

import os
os.chdir('/content/drive/MyDrive/IsotopePINN')
print(f'Working directory: {os.getcwd()}')

!PINN_FLOAT64=1 PINN_AMP=0 PINN_COMPILE=0 PINN_JOINT_CHUNK=0 PINN_LOG_EVERY=100 python train.py

In [ ]:
# ── Cell 6: Download results ──
# After training completes, this cell packages all outputs into a single zip
# and triggers a browser download.

import os, zipfile
from google.colab import files

PROJECT_DIR = '/content/drive/MyDrive/IsotopePINN'
OUTPUT_ZIP = '/content/IsotopePINN_Results.zip'

# Files to download
result_files = [
    'weights/pinn_best_weights.pth',
    'weights/pinn_trained_weights.pth',
    'graphs/pinn_loss_history.png',
    'graphs/loss_components.png',
    'graphs/pinn_ac225_pred_vs_true.png',
    'graphs/pinn_ac225_pred_vs_true_virgin_ic.png',
    'results/loss_history.csv',
    'results/last_training_run.json',
    'results/graph_manifest.json',
    'data/pinn_validation_summary.csv',
]

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for rel in result_files:
        full = os.path.join(PROJECT_DIR, rel)
        if os.path.exists(full):
            zf.write(full, rel)
            print(f'  ✅ {rel}')
        else:
            print(f'  ⚠️ {rel} (not found, skipping)')

print(f'\n📦 Results packaged: {OUTPUT_ZIP}')
print('📥 Downloading to your browser...')
files.download(OUTPUT_ZIP)